# PornWorks Web Generator — Kaggle Notebook

This notebook creates a temporary prompt-to-image web UI using the PornWorks Real Porn V03 SDXL checkpoint.

## Kaggle setup
1. Open Kaggle and create a new notebook: https://www.kaggle.com/code
2. In Notebook options, enable **Internet** and select an available **GPU** accelerator (T4 x2 or P100, if offered).
3. Upload this `.ipynb` file to Kaggle, or paste its cells into a new notebook.
4. Run the setup cell. When it finishes, restart the notebook session/kernel.
5. Run the remaining cells in order and open the temporary `gradio.live` link.

Kaggle commonly allows up to 12 hours per GPU notebook session, but GPU quotas, availability, and session limits can vary. This is not a permanent website; the share link stops when the session ends. Use only fictional consenting adults.

In [ ]:
# Replace incompatible optional quantization package and pin compatible library versions
!pip uninstall -y torchao diffusers transformers
!pip -q install "diffusers==0.31.0" "transformers==4.44.2" "huggingface-hub<1.0" "accelerate==0.34.2" safetensors gradio

In [ ]:
import importlib.util
if importlib.util.find_spec("torchao") is not None:
    raise RuntimeError("torchao is still installed. Run setup again, then restart the session/kernel.")

import torch
import gradio as gr
from diffusers import DiffusionPipeline

MODEL_ID = "John6666/pornworks-real-porn-v03-sdxl"
has_cuda = torch.cuda.is_available()
dtype = torch.float16 if has_cuda else torch.float32
print("GPU available:", has_cuda)
if has_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected; switch the Kaggle accelerator to GPU.")

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, use_safetensors=True)
if has_cuda:
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
else:
    pipe.to("cpu")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Model loaded. Scheduler:", pipe.scheduler.__class__.__name__)
print("VAE dtype:", next(pipe.vae.parameters()).dtype)

In [ ]:
import random
import numpy as np
from pathlib import Path

def generate_image(prompt, negative_prompt, width, height, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt first.")
    actual_seed = int(seed)
    if actual_seed < 0:
        actual_seed = random.randint(0, 2**31 - 1)
    generator = torch.Generator(device="cuda" if has_cuda else "cpu").manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            result = pipe(prompt=prompt, negative_prompt=negative_prompt or None,
                          width=int(width), height=int(height),
                          num_inference_steps=int(steps), guidance_scale=float(guidance),
                          generator=generator, output_type="pil")
        image = result.images[0].convert("RGB")
        arr = np.asarray(image)
        if arr.size == 0 or not np.isfinite(arr).all() or arr.std() < 1.0:
            raise gr.Error("The model returned a blank/invalid image. Check logs and restart the session.")
        path = f"/kaggle/working/generated_{actual_seed}.png" if Path('/kaggle/working').exists() else f"/content/generated_{actual_seed}.png"
        image.save(path, format="PNG")
        return image, path, f"Seed: {actual_seed} | Size: {image.width}×{image.height}"
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"), title="PornWorks Image Lab") as demo:
    gr.Markdown("# PornWorks Image Lab\nPrompt-to-image demo using the Real Porn V03 SDXL model. Use fictional consenting adults only.")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", placeholder="Describe the scene, lighting, camera and style…", lines=5)
            negative = gr.Textbox(label="Negative prompt", value="pixelated, low quality, blurry, jpeg artifacts, distorted anatomy, extra fingers, watermark, text", lines=2)
            width = gr.Slider(512, 1024, value=768, step=64, label="Width")
            height = gr.Slider(512, 1024, value=768, step=64, label="Height")
            steps = gr.Slider(20, 50, value=32, step=1, label="Steps")
            guidance = gr.Slider(1, 10, value=5.0, step=0.5, label="Guidance scale")
            seed = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
            run = gr.Button("Generate image", variant="primary")
        with gr.Column():
            output = gr.Image(label="Generated image", type="pil", format="png")
            file_output = gr.File(label="Download PNG")
            info = gr.Textbox(label="Generation info", interactive=False)
    run.click(generate_image, inputs=[prompt, negative, width, height, steps, guidance, seed], outputs=[output, file_output, info])

demo.queue(max_size=2).launch(share=True, debug=True)

## Test prompt
`A photorealistic red apple on a wooden table, natural window light, realistic shadows, sharp focus, detailed texture, professional photography`

If you still get corrupted pixels, copy the full generation error/log plus the printed GPU name and scheduler. Do not leave multiple GPU sessions running when finished.